In [11]:
# 0. Cargamos datos

import pandas as pd
import scipy.stats as st
from statsmodels.stats.proportion import proportions_ztest

df_demo    = pd.read_csv('Datasets_limpios/clean_client_profiles.csv')
df_web     = pd.read_csv('Datasets_limpios/clean_web_data.csv')
df_experiment = pd.read_csv('Datasets_limpios/clean_experiment_clients.csv')

In [12]:
# unimos los 3 dataframes

# Unimos web data con experiment para saber quién es Control y quién es Test
df = df_web.merge(df_experiment, on='client_id', how='inner')

# Añadimos los datos demográficos
df = df.merge(df_demo, on='client_id', how='inner')

print(df.shape)
print(df.columns.tolist())
df.head()

(317235, 14)
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time', 'variation', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth']


,client_id,visitor_id,visit_id,process_step,date_time,variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0


In [13]:
df['variation'].unique()

array(['Test', 'Control'], dtype=object)

In [14]:
# 1. Calcular completation rate (tasa de finalización)

# Separamos los grupos de la columna variation
control = df[df['variation'] == 'Control']
test    = df[df['variation'] == 'Test']

# Función para calcular el rate
def completion_rate(grupo):
    total       = grupo['client_id'].nunique() # cuenta cuantos clientes únicos hay en en el grupo
    completaron = grupo[grupo['process_step'] == 'confirm']['client_id'].nunique() # de esos clientes, cuántos llegaron hasta el pason confirm
    return completaron / total # divide los clienten que completaron entre el total del clientes

# Calcular tasas
tasa_control = completion_rate(control)
tasa_test    = completion_rate(test)

print(f"Completion rate Control: {tasa_control:.2%}")
print(f"Completion rate Test:    {tasa_test:.2%}")

Completion rate Control: 65.59%
Completion rate Test:    69.29%


In [15]:
# Guardar los 4 números para el z-test
total_control    = control['client_id'].nunique()
total_test       = test['client_id'].nunique()
completed_control = control[control['process_step'] == 'confirm']['client_id'].nunique()
completed_test    = test[test['process_step'] == 'confirm']['client_id'].nunique()

print(f"Control: {completed_control} completaron de {total_control} totales")
print(f"Test:    {completed_test} completaron de {total_test} totales")

Control: 15434 completaron de 23532 totales
Test:    18687 completaron de 26968 totales


Estos números nos dicen lo siguiente:
- En el grupo Control, el 65,59% de usuarios completo el proceso.
- En el grupo Test, el 69,29% de usuarios completó el proceso (casi 4 puntos más que el otro grupo).

Esto ahora toca demostrarlo, que esta diferencia de puntos no es azar. Para ello, usaremos el two-sided z-test porque vamos a comparar proporciones es decir 65% vs 69% de los grupos respectivamente.

**Recordatorio**

H₀ = hipótesis nula → la que asume que no pasa nada, que todo es igual

H₁ = hipótesis alternativa → la que dice que sí hay diferencia

In [ ]:
# 2. Hipotesis 1 (Hay diferencia entre el nuevo (Test) y antiguo (Control) diseño?)

#H0: tasa_test = tasa_control
# El completion rate del Test group (new) es igual a la del  Control group (old)
#H1: tasa_test ≠ tasa_control
# El completion rate del Test group (new) no es igual a la del Control group (old)

# Los datos que necesita el z-test
counts       = [completed_control, completed_test] # cuantos completaron en cada grupo
total_groups = [total_control, total_test] # cuántos usuarios  había en total en cada grupo

# Two-sided = dos colas = hay diferencia en cualquier dirección (mejor o peor)? es significativa?
stat, p_value = proportions_ztest(counts, total_groups, alternative='two-sided')
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("Rechazamos H0: la diferencia en completion rate es estadísticamente significativa")
else:
    print("No rechazamos H0: no hay evidencia suficiente de diferencia")

p-value: 0.0000
Rechazamos H0: la diferencia en completion rate es estadísticamente significativa


Aplicamos un two-sided  z-test para comparar la completion rate del grupo Control (65.59%) con la del grupo Test (69.29%). El test devuelve un p-value inferior 
a 0.05, por lo que rechazamos la hipótesis nula y concluimos que la diferencia en completion rate entre los dos grupos es estadísticamente significativa e imposible que sea casualidad.

Por tanto, se puede decir que  el nuevo diseñode la web hace que más gente termine el proceso. El resultado nos dice que sí, el grupo que usó el diseño nuevo completó un 69% de las veces, frente al 66% del diseño antiguo. Y esta diferencia no es casualidad, los números lo confirman.

In [ ]:
# 3. Hipotesis 2 (con umbral del 5% de coste-efectividad) se justifica el coste?
# H0: tasa_test ≤ tasa_control + 5%
# La tasa (completion rate) del grupo Test es menor o igual que la del grupo Control incrementada un 5%. Es decir no es suficientemente mejor 
# H1: tasa_test > tasa_control + 5%
# La tasa (completion rate) del grupo Test es mayor que la del grupo Control incrementada un 5%

In [ ]:
# Calculamos el umbral de negocio de Vanguard a la tasa del Control aumentada un 5%
umbral = tasa_control + 0.05

# One-sided = una cola = solo miramos si el Test es mayor
stat, p_value = proportions_ztest(
    [completed_control, completed_test],
    [total_control, total_test],
    value=0.05, # umbral estadístico, no confundir con el umbral de Vanguard que en este caso coinciden
    alternative='larger') # con esto se le dice al test que mire en una dirección, solo si es mayor, no si es menor

print(f"Tasa Control:    {tasa_control:.2%}")
print(f"Tasa Test:       {tasa_test:.2%}")
print(f"Umbral (tasa_control+ 5%):   {umbral:.2%}")
print(f"p-value:         {p_value:.4f}")

if p_value < 0.05:
    print("Rechazamos H0: el nuevo diseño supera el umbral del 5% — es coste-efectivo")
else:
    print("No rechazamos H0: el nuevo diseño no supera el umbral del 5%")

Tasa Control:    65.59%
Tasa Test:       69.29%
Umbral (tasa_control+ 5%):   70.59%
p-value:         1.0000
No rechazamos H0: el nuevo diseño no supera el umbral del 5%


Insgights:
- La tasa (completion rate) del Test group es 69,29%
- El umbral de negocio a superar que puso Vanguard es de 70,59% (Tasa Control 65,59% + 5%). Lo hace a
- El Test group no llega al umbral requerido, se queda 1.3 puntos por debajo (70,59% - 69,29%)
- P-value = 1.0 → no hay evidencia de que supere el umbral, no rechazamos hipotesis nula (H0).

Por tanto:

El p-value es mayor que 0.05, por lo tanto no podemos rechazar H0. El nuevo diseño no alcanza el umbral de coste-efectividad del 5% establecido por Vanguard.

El nuevo diseño sí mejora la tasa de finalización (completion rate), pero no lo suficiente como para justificar los costes. Vanguard necesitaba una mejora mínima del 5% y el nuevo diseño solo consigue un 3.7% de mejora (69,29% - 65,59%)


In [8]:
# Experiment evaluation
total = 23532 + 26968
print(f"Control: {23532/total:.2%}")
print(f"Test:    {26968/total:.2%}")


Control: 46.60%
Test:    53.40%


In [16]:
print(df.groupby('variation')['clnt_age'].mean())

variation
Control    48.284478
Test       48.722216
Name: clnt_age, dtype: float64


In [17]:
print(df.groupby('variation')['gendr'].value_counts(normalize=True))

variation  gendr
Control    U        0.344769
           M        0.337488
           F        0.317743
Test       M        0.335641
           U        0.335007
           F        0.329306
           X        0.000045
Name: proportion, dtype: float64


### Design Effectiveness

Total Control = 23.532 → 46.60%

Total Test = 26.968 → 53.40%

Total = 50.500

- **Was the experiment well-structured?**

El experimento sigue la estructura básica de un A/B test dos grupos, un diseño antiguo (Control) y uno nuevo (Test), y un proceso idéntico para ambos es decir, sgún el enunciado los 2 grupos (control y test) pasan por los siguientes pasos:
1. initial page
2. 3 subsequents steps (Step 1, 2 y 3)
3. Confirm

Además, tras realizar análisis sobre edad, género, balance económico y antigüedad de los clientes, pudimos observar que ambos grupos se encontraban bastante equilibrados. Además, el tamaño de la muestra es lo suficientemente grande como para realizar un análisis fiable y obtener resultados representativos ( n > 5.000).

La única diferencia era el diseño visual, uno vió el antiguo y otro el nuevo. Y esto es relevante porque significa que cualquier diferencia en los resultados se debe al diseño, no al proceso.

Por tanto, en términos generales se puede decir que está bien estructurado.


- **Were clients randomly and equally divided between the old and new designs?**

La división entre grupos parece equilibrada ya que un  46.6% de los participantes vieron el diseño antiguo (Control group) y un 53.4% el nuevo (Test group). 

Además, al limpiar los datos encontramos valores nulos NaN en la columna 'variation', es decir, clientes que no tenían grupo asignado (Control/Test). Tuvimos que eliminarlos del análisis y no sabemos si habrían afectado a los resultados de uno u otro grupo.

En general podemos decir que los analisis demograficos y financieros sugieren que los clientes fueron distribuidos de manera muy equilibrada entre ambos grupos.


- Were there any biases?

Sí, el principal sesgo detectado es la eliminación de clientes sin grupo asignado, lo que provocó un desequilibrio entre grupos (46.6% Control vs 53.4% Test). Además, al no saber cómo se asignaron los clientes, no podemos descartar que haya habido algún sesgo en la selección.

Sin embargo, la edad media de ambos grupos es muy similar (Control 48.28 años vs Test 48.72 años), lo que sugiere que el reparto de perfiles de clientes fue bastante equilibrado.

Durante el análisis también identificamos algunas limitaciones, como el elevado número de valores “Unknown” en la variable género y la presencia de outliers, especialmente en variables relacionadas con el balance monetario y el tiempo entre pasos; obteniendo como tiempo máximo en una visita aprox. unas 11 horas. Lo cual podemos interpretar que el cliente dejara la sesión abierta sin terminarla.

Aun así, no se detectaron desequilibrios importantes que comprometan la validez general del experimento.


### Duration Assessment
- Was the timeframe of the experiment (from 3/15/2017 to 6/20/2017) adequate to gather meaningful data and insights?

No del todo, el experimento duró 3 meses aprox (15 marzo - 20 junio 2017) y aunque es suficiente para obtener una primera idea, no cubre todos los ciclos de comportamiento de los clientes.

En una empresa de inversiones, probablemente los clientes suelen ser más activos en ciertos meses como por ejemplo al principio del año cuando se planifican presupuestos, o a final de año por temas fiscales. Al cubrir solo un periodo, el experimento podría no reflejar el comportamiento real de los clientes durante todo el año.

Idealmente, el experimento debería haberse extendido al menos 6 o  año completo, o haberse realizado en los meses de mayor y menor actividad para tener una visión más completa.

### Additional Data Needs
- What other data, if available, could enhance the analysis?

Motivo de abandono: conocer en qué paso exacto salieron los clientes que no completaron el proceso ayudaría a identificar los puntos de fricción

Creo que otros datos que podrían enriquecer el análisis serían la satisfacción del cliente, por ejemplo a través de encuestas o surveys. También sería interesante contar con información sobre la ubicación de los usuarios (país o región) para analizar si la nueva interfaz funciona igual en todas las localizaciones. Además, disponer de datos sobre errores técnicos ayudaría a detectar posibles problemas y mejorar la experiencia de la nueva interfaz en prueba
